# Part 14 — Context-Aware Chunking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mftnakrsu/rag-by-hand/blob/main/part-14-context-aware-chunking/context_aware_chunking.ipynb)

*RAG from First Principles. Part 14 of the series, on the Frontier Track: two training-free ways to stop a chunk from losing its document context.*

📖 Read the essay: https://www.mefby.com/essays/context-aware-chunking

🐍 Companion script: `context_aware_chunking.py`

## What you'll build

Back in Part 5 we cut documents into chunks. The catch nobody mentions: a chunk that reads perfectly in context can become *uninterpretable* once it stands alone. Split a refund-policy note into sentences and one chunk reads "She set the refund window at 30 days." Who is "she"? Embed that line by itself and the vector never sees "Alice" or "the refund policy" — it is blind to the very context that makes the chunk answerable. A query that names Alice then can't find the one chunk that answers it.

This part builds, by hand, the two **training-free** fixes:

- **Late chunking** (introduced by the Jina AI team): embed *all* the document's tokens in one pass through a long-context encoder, **then** mean-pool each chunk's token span into a chunk vector. Because the pooling happens *after* the transformer, every chunk vector is already contextualized by the rest of the document. This is *not* Part 5's static title-prepend — nothing is glued onto the text.
- **Contextual Retrieval** (Anthropic): for each chunk, a small LLM writes a one-sentence situating note ("This chunk is from the 2023 refund policy...") and you **prepend it before embedding**. Model-agnostic and drop-in; it costs one extra LLM call per chunk at index time (prompt caching mitigates that).

We will watch the buried answer chunk surface under each fix on the same coreference query, then compare the two.

> **Runs fully offline.** Every code cell executes with **no network and no API key** — numpy only. The real long-context encoder is the headline (the code you'd actually ship); if a model isn't available, a transparent deterministic stand-in keeps the cell runnable and prints a clear label when it kicks in.

## Setup

Two imports: `re` for tokenizing and `numpy` for the vector math. `hashlib` powers the deterministic offline embedder. Nothing here touches the network.

In [ ]:
import hashlib
import re

import numpy as np

print("numpy", np.__version__, "— ready (no network, no API key required)")

## Step 0 — The document, written as a coreference trap

Our running example stays continuous with Parts 6–12: a short refund-policy note. It is written so the *answer* sentence is a coreference trap. Chunk `[1]` is the one that actually answers "what is Alice's refund window?", but on its own it says **"She"**, not "Alice", so a naive per-chunk embedding will bury it. Chunk `[3]` reuses the `E-4042` error code from the running support knowledge base.

In [ ]:
DOCUMENT = [
    "Alice founded Acme in 2019.",
    "She set the refund window at 30 days.",
    "Returns outside that window are declined automatically.",
    "The error code E-4042 means the window has already closed.",
]
QUERY = "what is Alice's refund window?"
# A human-readable title used by the offline situating-note writer.
DOC_TITLE = "a note about Alice and Acme's refund policy"

for i, ch in enumerate(DOCUMENT):
    print(f"  [{i}] {ch}")
print(f'\nQuery: "{QUERY}"')
print("The answer is chunk [1] — but it says 'She', not 'Alice'.")

## Step 1 — Tokenizing

We stay at the level of plain lowercased words, exactly like the earlier parts. These token boundaries *are* the span boundaries that late chunking will pool over, so we keep the tokenizer dead simple and shared everywhere.

In [ ]:
_TOKEN_RE = re.compile(r"[a-z0-9]+")


def tokenize(text):
    return _TOKEN_RE.findall(text.lower())


print(tokenize("She set the refund window at 30 days."))

## Step 2 — Small geometry helpers

Everything downstream compares meaning as *closeness in a shared vector space* (Parts 2–3). We need row-wise L2 normalization (so a dot product reads straight off as cosine) and a plain cosine similarity.

In [ ]:
def _l2_normalize_rows(mat):
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return mat / norms


def cosine(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return 0.0 if denom == 0 else float(np.dot(a, b) / denom)


print("geometry helpers ready")

## Step 3 — A token embedder that *contextualizes*

The intended path is a **long-context encoder** that returns a vector **per token**, each contextualized by the whole input. That contextualization is the whole point: it is what lets pooling a span *after* a full-document pass beat embedding the chunk alone.

**Why the offline fallback must contextualize too.** A pure per-token hash carries no neighbour information, so pooling a chunk's span out of a whole-document pass would give the *exact same* vector as embedding the chunk on its own — and late chunking would be indistinguishable from naive chunking. A real transformer avoids that with **attention**: each token's output vector mixes in its neighbours. We mimic that transparently — a token's vector is its own hashed direction *plus* a distance-decayed blend of the surrounding tokens' hashes (a toy "attention window"). It still can't make a hash understand that "she" means "alice", but it faithfully reproduces the mechanism: run it on the whole document and the "she" token picks up the nearby "alice"/"refund" directions; run it on the lone chunk and it does not.

In [ ]:
FALLBACK_DIM = 256       # small, deterministic; mirrors a fixed-length vector
_CONTEXT_DECAY = 0.55    # how strongly a token absorbs its neighbours (toy attention)


def _hashed_token_vector(token, dim):
    """A reproducible pseudo-random unit direction for a single token."""
    h = hashlib.sha256(token.encode("utf-8")).digest()
    raw = np.frombuffer((h * (dim // len(h) + 1))[:dim], dtype=np.uint8)
    return (raw.astype(np.float64) / 255.0) * 2.0 - 1.0


def _fallback_token_embed(text):
    """Per-token, CONTEXTUALIZED vectors with NO model.

    Step 1: hash each token to its own raw direction.
    Step 2: contextualize — each token's vector becomes itself plus a
    distance-decayed sum of every OTHER token's raw direction in the same input.
    This depends on the WHOLE input, so embedding the 'she' chunk inside the full
    document yields a different vector than embedding that chunk alone — exactly
    what late chunking relies on.
    """
    toks = tokenize(text)
    n = len(toks)
    if n == 0:
        return np.zeros((0, FALLBACK_DIM))
    raw = np.vstack([_hashed_token_vector(t, FALLBACK_DIM) for t in toks])
    out = raw.copy()
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            out[i] += (_CONTEXT_DECAY ** abs(i - j)) * raw[j]
    return _l2_normalize_rows(out)


# Proof the fallback contextualizes: the 'she' token's vector differs when it is
# embedded alone vs inside the whole document.
alone = _fallback_token_embed("She set the refund window at 30 days.")[0]
in_doc = _fallback_token_embed(" ".join(DOCUMENT))[len(tokenize(DOCUMENT[0]))]
print(f"cosine('she' alone vs 'she' in the full document) = {cosine(alone, in_doc):.3f}")
print("  < 1.0  -> the token vector really does change with its context")

## Step 4 — The real long-context encoder, behind a transparent fallback

This is the **headline**: the code you'd actually ship. We pull per-token vectors from `all-MiniLM-L6-v2`'s underlying transformer (`last_hidden_state`), dropping the `[CLS]`/`[SEP]` specials so token spans line up with words. We wrap the whole load in `try/except` and **touch it once** so a missing-weights / offline failure happens *inside the guard* — then drop to the deterministic stand-in and print which path is live. Many sentence encoders only expose a pooled sentence vector, so reaching for `last_hidden_state` is what gives us genuine per-token vectors.

In [ ]:
def load_token_embedder():
    """Return (token_embed_fn, using_real_bool)."""
    try:
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer("all-MiniLM-L6-v2")
        tok = model.tokenizer
        transformer = model[0].auto_model        # the raw HF transformer

        import torch

        device = next(transformer.parameters()).device   # cpu or cuda

        def token_embed(text):
            enc = tok(text, return_tensors="pt", truncation=True)
            enc = {k: v.to(device) for k, v in enc.items()}   # match model's device
            with torch.no_grad():
                out = transformer(**enc)
            hidden = out.last_hidden_state[0].cpu().numpy()   # (n_tokens, d)
            if hidden.shape[0] >= 2:
                hidden = hidden[1:-1]                          # drop [CLS]/[SEP]
            return _l2_normalize_rows(hidden)

        _ = token_embed("warm up")        # force a missing-weights failure HERE
        return token_embed, True
    except Exception as exc:
        print(f"[real long-context encoder unavailable: {type(exc).__name__}] "
              "-> using offline hashing fallback")
        return _fallback_token_embed, False


token_embed, USING_REAL = load_token_embedder()
label = ("REAL long-context encoder (all-MiniLM-L6-v2 token states)"
         if USING_REAL else "offline hashing fallback  (no model, no network)")
print(f"Embedder in use: {label}")

## Step 5 — Embedding a string, and ranking chunks

`embed_string` is the **naive** path: pool a single string's token vectors into one chunk vector, so its tokens only ever see the chunk's own words (no document context). `rank_by_cosine` scores every chunk vector against the query and sorts best-first.

In [ ]:
def embed_string(text):
    """Pool one string's token vectors into a single chunk vector (NAIVE path)."""
    tv = token_embed(text)
    if tv.shape[0] == 0:
        return np.zeros(FALLBACK_DIM)
    v = tv.mean(axis=0)
    return v / (np.linalg.norm(v) + 1e-9)


def rank_by_cosine(query_vec, chunk_vecs):
    """Return [(chunk_index, cosine), ...] sorted best-first."""
    scored = [(i, cosine(query_vec, cv)) for i, cv in enumerate(chunk_vecs)]
    scored.sort(key=lambda pair: -pair[1])
    return scored


def show_ranking(ranking):
    for rank, (i, score) in enumerate(ranking, start=1):
        preview = DOCUMENT[i] if len(DOCUMENT[i]) <= 52 else DOCUMENT[i][:50] + "..."
        print(f"  {rank:>2}.  {score:.3f}  [{i}] {preview}")
    return [i for i, _ in ranking].index(1) + 1   # 1-based rank of answer chunk [1]


query_vec = embed_string(QUERY)
print("query embedded; ready to rank")

## Step 6 — Naive chunking: embed each chunk on its own

The baseline. Embed every chunk string independently, then rank by cosine to the query. Watch where the answer chunk `[1]` lands — on the offline (weaker) embedder it gets buried below an off-topic chunk that merely shares the word "window", because "She" has lost its antecedent "Alice".

In [ ]:
naive_vecs = [embed_string(ch) for ch in DOCUMENT]
naive_rank = show_ranking(rank_by_cosine(query_vec, naive_vecs))
if naive_rank > 1:
    print(f"\n  -> answer chunk [1] BURIED at rank {naive_rank}: 'She' lost 'Alice',")
    print("     so alone it can't out-rank a chunk that only shares 'window'.")
else:
    print("\n  -> this encoder already ranks [1] first; the trap bites harder on")
    print("     longer docs and weaker encoders (see the offline fallback path).")

## Step 7 — Late chunking: embed all tokens once, then pool the spans

Here is the correctness-critical function. The key move: feed in token vectors that were produced from the **whole document in one pass**, then pool each chunk's `(start, end)` span. Because the token vectors already mixed in their neighbours, every pooled chunk vector is contextualized by the rest of the document — *same chunk text, contextualized vector*.

In [ ]:
def late_chunk(token_vecs: np.ndarray, spans: list[tuple[int, int]]) -> np.ndarray:
    """Pool token spans into chunk vectors AFTER the encoder, so each chunk
    vector is contextualized by the whole document. spans are (start, end)
    token indices. Returns (n_chunks, d), L2-normalized."""
    out = []
    for s, e in spans:
        v = token_vecs[s:e].mean(axis=0)
        out.append(v / (np.linalg.norm(v) + 1e-9))
    return np.array(out)


def chunk_spans(chunks):
    """Embed the WHOLE document once, then record each chunk's token span."""
    full_text = " ".join(chunks)
    full_vecs = token_embed(full_text)
    spans, cursor = [], 0
    for ch in chunks:
        n = len(tokenize(ch))
        spans.append((cursor, cursor + n))
        cursor += n
    # A real sub-word tokenizer may emit a different token count than our word
    # tokenizer; fall back to proportional spans so we never index out of range.
    if cursor != full_vecs.shape[0] and full_vecs.shape[0] > 0:
        total = cursor
        spans, cursor = [], 0
        for ch in chunks:
            frac = len(tokenize(ch)) / max(total, 1)
            n = max(1, round(frac * full_vecs.shape[0]))
            spans.append((cursor, min(cursor + n, full_vecs.shape[0])))
            cursor = min(cursor + n, full_vecs.shape[0])
        spans[-1] = (spans[-1][0], full_vecs.shape[0])
    return full_vecs, spans


full_vecs, spans = chunk_spans(DOCUMENT)
late_vecs = late_chunk(full_vecs, spans)
late_rank = show_ranking(rank_by_cosine(query_vec, late_vecs))
print("\n  -> answer chunk [1] now ranks FIRST: pooling its span AFTER the whole-")
print("     document pass folds the surrounding 'alice'/'refund' context into it.")

## Step 8 — Contextual Retrieval: prepend an LLM situating note, then embed

The second fix needs no long-context encoder. For each chunk, a small LLM writes a one-sentence situating note and you **prepend it before embedding**. The intended path calls an LLM with the chunk plus the whole document; offline we use a deterministic, grounded `generate()` stand-in that templates the note from the document title only — never inventing facts. The mechanism (prepend-then-embed) is identical; only the wording quality differs.

In [ ]:
def generate(prompt, doc_title):
    """Grounded offline stand-in for an LLM situating-note writer.

    A real call would be roughly:
        client.chat.completions.create(model=..., messages=[{...prompt...}])
    Here we deterministically template the note from the document title.
    """
    return f"This chunk is from {doc_title}."


def contextualize(chunk, doc_title):
    """Return the situating note an LLM would prepend to this chunk."""
    prompt = ("Write one short sentence situating this chunk within the document.\n"
              f"Document: {doc_title}\nChunk: {chunk}\nSituating sentence:")
    return generate(prompt, doc_title)


note = contextualize(DOCUMENT[1], DOC_TITLE)
print(f'Situating note for chunk [1]:  "{note}"')
print("Embedded text = situating-note + chunk, encoded as one string.\n")

ctx_vecs = [embed_string(contextualize(ch, DOC_TITLE) + " " + ch) for ch in DOCUMENT]
ctx_rank = show_ranking(rank_by_cosine(query_vec, ctx_vecs))
print("\n  -> answer chunk [1] ranks FIRST again: the note carries 'Alice' and")
print("     'refund policy' into the chunk's own string before the embedder sees it.")

## Step 9 — Scoreboard and the comparison

Side by side: where does the answer chunk `[1]` rank under each strategy? Then the honest comparison — late chunking needs a long-context encoder but **no LLM**; contextual retrieval is **model-agnostic** but costs an LLM call per chunk at index time. They compose.

In [ ]:
print("rank of the answer chunk [1] under each strategy (1 = best):\n")
naive_note = ("orphaned 'She' -> buried below an off-topic chunk"
              if naive_rank > 1 else "this encoder resolves 'She' even in isolation")
print(f"  naive chunking          : rank {naive_rank} of {len(DOCUMENT)}   ({naive_note})")
print(f"  late chunking           : rank {late_rank} of {len(DOCUMENT)}   (pool spans after the encoder)")
print(f"  contextual retrieval    : rank {ctx_rank} of {len(DOCUMENT)}   (prepend a situating note)")
print()
print("                       | late chunking        | contextual retrieval")
print("  ---------------------+----------------------+----------------------")
print("  needs an LLM?        | no                   | yes (1 call / chunk)")
print("  needs long-context   | yes                  | no (model-agnostic)")
print("   encoder?            |                      |")
print("  index-time cost      | one encoder pass     | one LLM call per chunk")
print("  composable?          | yes — use both       | yes — use both")

## What you learned

- A chunk that reads fine in isolation can be **uninterpretable** once it leaves its document: "She" loses "Alice", "the policy" loses its antecedent. Embedded alone, that chunk is invisible to a query that names what it dropped (Part 5's chunking, revisited).
- **Late chunking** fixes this with no LLM: embed *all* the document's tokens in one pass through a long-context encoder, **then** pool each chunk's token span. The pooling happens *after* the transformer, so each chunk vector is already contextualized by the whole document. Unlike Part 5's static title-prepend, nothing is glued onto the text.
- **Contextual Retrieval** fixes it with no special encoder: an LLM writes a one-sentence situating note per chunk and you **prepend it before embedding**. Model-agnostic and drop-in; the cost is one extra LLM call per chunk at index time (prompt caching mitigates it).
- The one defensible number: contextual embeddings **alone** cut the top-20 retrieval-failure rate by **35%** (5.7% → 3.7%, Anthropic). Larger cumulative figures bundle in BM25 and reranking, so they overstate the chunking-only effect — we don't quote them.
- The two are **composable**: late-chunk first, then contextualize the spans, if you have both a long-context encoder and an LLM budget.
- Everything ran **fully offline**: the real long-context encoder is the headline, with a transparent contextualizing stand-in so every cell executes with no network and no API key.

### Next

**Part 15 — Adaptive RAG** closes the Frontier Track: a small complexity classifier that routes each query to no-retrieval, single-step, or multi-step retrieval — so you don't pay for machinery an easy query doesn't need, and don't under-serve a hard one. (Previous: **Part 13 — Late-Interaction Retrieval**.)